In [1]:
import os
# This will find every CSV file in your input folder and print the full path
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/konstantineendeladze/iee-fraud-detection/sample_submission.csv
/kaggle/input/datasets/konstantineendeladze/iee-fraud-detection/test_identity.csv
/kaggle/input/datasets/konstantineendeladze/iee-fraud-detection/train_identity.csv
/kaggle/input/datasets/konstantineendeladze/iee-fraud-detection/test_transaction.csv
/kaggle/input/datasets/konstantineendeladze/iee-fraud-detection/train_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import pandas as pd

# Use the official competition path found by your script
path = "/kaggle/input/competitions/ieee-fraud-detection/"

# Load the training data
train_transaction = pd.read_csv(path + "train_transaction.csv")
train_identity = pd.read_csv(path + "train_identity.csv")

test_transaction = pd.read_csv(path + "train_transaction.csv")
test_identity = pd.read_csv(path + "train_identity.csv")


# Merge them together on TransactionID
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

print(f"Success! Final dataframe shape: {train.shape}")
print(train.head())

Success! Final dataframe shape: (590540, 434)
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceTy

In [3]:
import numpy as np
import pandas as pd

def reduce_memory(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory before: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory after: {end_mem:.2f} MB")
    
    return df

In [4]:
train = reduce_memory(train)
test = reduce_memory(test)

Memory before: 1955.37 MB
Memory after: 1044.70 MB
Memory before: 1955.37 MB
Memory after: 1044.70 MB


In [5]:
missing_ratio = train.isnull().mean()

drop_cols = missing_ratio[missing_ratio > 0.99].index
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

In [6]:
constant_cols = [col for col in train.columns if train[col].nunique() <= 1]

train = train.drop(columns=constant_cols)
test = test.drop(columns=constant_cols)

In [7]:
y = train["isFraud"]
X = train.drop(columns=["isFraud"])

X, test = X.align(test, join="left", axis=1)
out_path = "/kaggle/working/"
X.to_parquet(out_path + "X_clean.parquet")
y.to_csv(out_path + "y.csv", index=False)
test.to_parquet(out_path + "test_clean.parquet")